# News Sentiment Analysis

Scrape NEPSE news headlines from Sharesansar, score sentiment, and explore linking sentiment to individual stocks.

In [13]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import time
import warnings
warnings.filterwarnings('ignore')

print("All imports successful!")

All imports successful!


## 1. Scrape News Headlines

Scrapes Sharesansar's latest news feed using cursor-based pagination (`rel="next"` link) to avoid duplicate pages.

In [14]:
# ── Scrape 50 pages of news (fixed pagination) ───────────────
def scrape_news_page(driver, url):
    driver.get(url)
    time.sleep(2)
    soup = BeautifulSoup(driver.page_source, 'lxml')
    
    news_items = soup.find_all('div', class_='featured-news-list')
    articles = []
    for item in news_items:
        title_tag = item.find('h4', class_='featured-news-title')
        date_tag = item.find('span', class_='text-org')
        link_tag = item.find('a', href=lambda x: x and 'newsdetail' in x)
        if title_tag and link_tag:
            articles.append({
                'title': title_tag.get_text(strip=True),
                'date': date_tag.get_text(strip=True) if date_tag else None,
                'url': link_tag['href']
            })
    
    # Get next page cursor - specifically rel="next" to avoid loop
    next_link = soup.find('a', rel='next')
    next_cursor = next_link['href'] if next_link else None
    
    return articles, next_cursor


options = Options()
options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

all_articles = []
base_url = 'https://www.sharesansar.com/category/latest'
current_url = base_url

for page in range(50):
    articles, next_cursor = scrape_news_page(driver, current_url)
    all_articles.extend(articles)
    
    if (page + 1) % 10 == 0:
        print(f"Page {page+1}/50 | Total articles: {len(all_articles)}")
    
    if next_cursor:
        current_url = f"{base_url}{next_cursor}"
    else:
        print("No more pages")
        break
    
    time.sleep(0.5)

driver.quit()

print(f"\nTotal articles scraped: {len(all_articles)}")

df_news = pd.DataFrame(all_articles)
df_news = df_news.drop_duplicates(subset='url').reset_index(drop=True)
print(f"After removing duplicates: {len(df_news)}")

# Save to dedicated news folder
df_news.to_csv('../data/news/news_headlines.csv', index=False)
print("Saved to data/news/news_headlines.csv")
print(df_news.head())

Page 10/50 | Total articles: 100
Page 20/50 | Total articles: 200
Page 30/50 | Total articles: 300
Page 40/50 | Total articles: 400
Page 50/50 | Total articles: 500

Total articles scraped: 500
After removing duplicates: 500
Saved to data/news/news_headlines.csv
                                               title                    date  \
0  Just 16 Days Left: Nepal Has Spent Only 35% of...  Tuesday, June 30, 2026   
1  Government Appoints Three International Expert...  Tuesday, June 30, 2026   
2  NEPSE Plunges to Six-Month Low Level After Sev...  Tuesday, June 30, 2026   
3  IZML Recovers Millions in Overdue Rent Followi...  Tuesday, June 30, 2026   
4  Government Plans Public-Private Partnership to...  Tuesday, June 30, 2026   

                                                 url  
0  https://www.sharesansar.com/newsdetail/just-16...  
1  https://www.sharesansar.com/newsdetail/governm...  
2  https://www.sharesansar.com/newsdetail/nepse-p...  
3  https://www.sharesansar.com/newsd

## 2. Sentiment Scoring — TextBlob (baseline)

TextBlob is a general-purpose sentiment analyzer. As shown below, it fails on financial language — e.g. "NEPSE Plunges... After Seventh Straight Session of Losses" scores as near-neutral (0.10) instead of clearly negative.

In [15]:
from textblob import TextBlob

def get_sentiment(text):
    blob = TextBlob(text)
    polarity = blob.sentiment.polarity  # -1 (negative) to +1 (positive)
    
    if polarity > 0.1:
        label = 'Positive'
    elif polarity < -0.1:
        label = 'Negative'
    else:
        label = 'Neutral'
    
    return polarity, label


df_news['sentiment_score'] = df_news['title'].apply(lambda x: get_sentiment(x)[0])
df_news['sentiment_label'] = df_news['title'].apply(lambda x: get_sentiment(x)[1])

print(df_news[['title', 'sentiment_score', 'sentiment_label']].head(15))
print(f"\nSentiment distribution:")
print(df_news['sentiment_label'].value_counts())

                                                title  sentiment_score  \
0   Just 16 Days Left: Nepal Has Spent Only 35% of...        -0.033333   
1   Government Appoints Three International Expert...         0.000000   
2   NEPSE Plunges to Six-Month Low Level After Sev...         0.100000   
3   IZML Recovers Millions in Overdue Rent Followi...         0.000000   
4   Government Plans Public-Private Partnership to...         0.000000   
5   Know How Did The Mutual Funds Perform in Jesth...         0.050000   
6   Nabil Bank Announces Branch Mergers and Reloca...         0.000000   
7   Kathmandu Traffic Police Penalize 1,592 Vehicl...         0.000000   
8   Nepal Rastra Bank to Issue Rs. 35 Billion in C...         0.000000   
9   Prime Commercial Bank Signs Investment Deal fo...         0.000000   
10  Tundi Power Appoints Kumari Capital as Issue M...         0.000000   
11  Gold Falls by Rs. 4500 Today; Check Today's Tr...         0.000000   
12  Nepal SBI Bank Marks 33rd Annivers

## 3. Sentiment Scoring — FinBERT (financial domain model)

FinBERT is a BERT model fine-tuned specifically on financial text. It correctly classifies the same "NEPSE Plunges" headline as **negative with 95% confidence** — a major improvement over TextBlob.

In [16]:
from transformers import pipeline

finbert = pipeline("sentiment-analysis", model="ProsusAI/finbert", framework="pt")

test_headline = "NEPSE Plunges to Six-Month Low Level After Seventh Straight Session of Losses"
result = finbert(test_headline)
print(f"Headline: {test_headline}")
print(f"FinBERT result: {result}")

Headline: NEPSE Plunges to Six-Month Low Level After Seventh Straight Session of Losses
FinBERT result: [{'label': 'negative', 'score': 0.9543922543525696}]


In [17]:
# Apply FinBERT to all headlines
def get_finbert_sentiment(text):
    result = finbert(text)[0]
    return result['label'], result['score']

results = df_news['title'].apply(get_finbert_sentiment)
df_news['finbert_label'] = results.apply(lambda x: x[0])
df_news['finbert_score'] = results.apply(lambda x: x[1])

print(df_news[['title', 'finbert_label', 'finbert_score']].head(15))
print(f"\nFinBERT sentiment distribution:")
print(df_news['finbert_label'].value_counts())

                                                title finbert_label  \
0   Just 16 Days Left: Nepal Has Spent Only 35% of...       neutral   
1   Government Appoints Three International Expert...       neutral   
2   NEPSE Plunges to Six-Month Low Level After Sev...      negative   
3   IZML Recovers Millions in Overdue Rent Followi...      negative   
4   Government Plans Public-Private Partnership to...      positive   
5   Know How Did The Mutual Funds Perform in Jesth...       neutral   
6   Nabil Bank Announces Branch Mergers and Reloca...       neutral   
7   Kathmandu Traffic Police Penalize 1,592 Vehicl...      negative   
8   Nepal Rastra Bank to Issue Rs. 35 Billion in C...       neutral   
9   Prime Commercial Bank Signs Investment Deal fo...      positive   
10  Tundi Power Appoints Kumari Capital as Issue M...       neutral   
11  Gold Falls by Rs. 4500 Today; Check Today's Tr...      negative   
12  Nepal SBI Bank Marks 33rd Anniversary with Nat...      positive   
13  Ra

In [18]:
# Save sentiment data
df_news.to_csv('../data/news/news_sentiment.csv', index=False)
print("Saved to data/news/news_sentiment.csv")

# Try to match headlines to known tickers
with open('../data/processed/ticker_list.txt', 'r') as f:
    tickers = [line.strip() for line in f.readlines()]

# Simple company name to ticker mapping (we'll expand this)
def find_mentioned_ticker(title, tickers):
    title_upper = title.upper()
    for ticker in tickers:
        if ticker in title_upper.split():
            return ticker
    return None

df_news['mentioned_ticker'] = df_news['title'].apply(lambda x: find_mentioned_ticker(x, tickers))

matched = df_news[df_news['mentioned_ticker'].notna()]
print(f"\nHeadlines with ticker mentions: {len(matched)}/{len(df_news)}")
print(matched[['title', 'mentioned_ticker', 'finbert_label']].head(10))

Saved to data/news/news_sentiment.csv

Headlines with ticker mentions: 32/500
                                                 title mentioned_ticker  \
6    Nabil Bank Announces Branch Mergers and Reloca...            NABIL   
12   Nepal SBI Bank Marks 33rd Anniversary with Nat...              SBI   
99   NIC ASIA Bank Appoints NIMB ACE Capital as Rig...             NIMB   
121  Chhimek Laghubitta Elects Rishiraj Joshi as Bo...            JOSHI   
134  Nepal SBI Bank Ltd. Celebrates International Y...              SBI   
138  Sanima Reliance Life Insurance Wins “Corporate...           SANIMA   
141  Dhangadhi Sub-Metropolitan City unveils Rs 2.6...             CITY   
146  Himstar Becomes Co-Sponsor of FIFA World Cup 2...          HIMSTAR   
153  Nabil Bank Limited Maintains Strong Credit Rat...            NABIL   
230  ICRA Nepal Upgrades Jhapa Energy to ‘B+’ Ratin...            JHAPA   

    finbert_label  
6         neutral  
12       positive  
99        neutral  
121       neutra

## 4. Linking Headlines to Specific Stocks

Attempted to match headlines to tickers using company names from the NEPSE API. Two approaches were tried:
- **v1**: exact ticker symbol match — too restrictive (32/500 matched)
- **v2**: first word of company name match — too loose, produced false positives (e.g. "Kathmandu" matching ticker "MANDU", "Government" matching "PDB")

**Conclusion:** reliable per-stock news matching needs proper NER or a curated alias dictionary — out of scope for now. Documented here as a known limitation / future improvement.

In [19]:
from nepse import Nepse

nepse = Nepse()
nepse.setTLSVerification(False)
companies = nepse.getCompanyList()

# Build ticker -> company name mapping
ticker_to_name = {}
for c in companies:
    ticker_to_name[c['symbol']] = c['companyName']

# Show a few examples
for ticker in ['NABIL', 'SBI', 'NIMB', 'SANIMA']:
    print(f"{ticker}: {ticker_to_name.get(ticker)}")

NABIL: Nabil Bank Limited
SBI: Nepal SBI Bank Limited
NIMB: Nepal Investment Mega Bank Limited
SANIMA: Sanima Bank Limited


In [20]:
def find_mentioned_ticker_v2(title, ticker_to_name):
    title_lower = title.lower()
    matches = []
    
    for ticker, company_name in ticker_to_name.items():
        if not company_name:
            continue
        # Extract first 1-2 significant words of company name (e.g. "Nabil" from "Nabil Bank Limited")
        name_words = company_name.replace('Limited', '').replace('Ltd.', '').strip().split()
        if not name_words:
            continue
        first_word = name_words[0].lower()
        
        # Match if first significant word appears in title (avoid short generic words)
        if len(first_word) > 3 and first_word in title_lower:
            matches.append(ticker)
    
    return matches[0] if len(matches) == 1 else None  # only keep unambiguous single matches


df_news['mentioned_ticker_v2'] = df_news['title'].apply(
    lambda x: find_mentioned_ticker_v2(x, ticker_to_name)
)

matched_v2 = df_news[df_news['mentioned_ticker_v2'].notna()]
print(f"Headlines with ticker mentions (v2): {len(matched_v2)}/{len(df_news)}")
print(matched_v2[['title', 'mentioned_ticker_v2', 'finbert_label']].head(15))

Headlines with ticker mentions (v2): 58/500
                                                 title mentioned_ticker_v2  \
4    Government Plans Public-Private Partnership to...                 PDB   
7    Kathmandu Traffic Police Penalize 1,592 Vehicl...               MANDU   
13   Rasuwagadhi Hydropower Project Resumes Full Op...                RHPL   
15   Kajaria Ramesh Tiles Expands Kathmandu Presenc...               MANDU   
20   Ministry of Commerce to Withdraw Proposed 1% S...                 PDB   
23   Bonus Shares of Three Companies Have Been List...                TSHL   
25   Kalikot Declared Third Fully Vaccinated Distri...                KRBL   
38   Lawmakers Unanimously Approve Government Borro...                 PDB   
48   Paramount Motors Expands EV Portfolio, Set to ...               MMDBL   
54   Life and Non-Life Insurance Sectors Report Var...                LICN   
62   Conversion of Promoter Shares of Forward Micro...               FOWAD   
65   Devastating Twi

## 5. Market-Wide Daily Sentiment (the actually useful output)

Since per-stock matching is unreliable, we instead aggregate sentiment **per day across all headlines** — a single daily "market mood" score that can be merged into every sector model as one additional feature.

**Limitation:** only ~1 month of news history is available via this scraper (Sharesansar's feed doesn't go back further this way), so this isn't yet enough data to meaningfully retrain the 2-year stock models. The next step would be running this scraper daily going forward to accumulate history.

In [21]:
# Convert date strings to datetime
df_news['Date'] = pd.to_datetime(df_news['date'], format='%A, %B %d, %Y', errors='coerce')

# Map sentiment to numeric score: positive=1, neutral=0, negative=-1
sentiment_map = {'positive': 1, 'neutral': 0, 'negative': -1}
df_news['sentiment_numeric'] = df_news['finbert_label'].map(sentiment_map)

# Daily aggregate sentiment
daily_sentiment = df_news.groupby('Date').agg(
    avg_sentiment=('sentiment_numeric', 'mean'),
    news_count=('sentiment_numeric', 'count')
).reset_index()

print(daily_sentiment)
daily_sentiment.to_csv('../data/news/daily_market_sentiment.csv', index=False)
print("\nSaved to data/news/daily_market_sentiment.csv")

         Date  avg_sentiment  news_count
0  2026-05-30       0.000000           1
1  2026-05-31       0.142857          14
2  2026-06-01       0.105263          19
3  2026-06-02       0.041667          24
4  2026-06-03       0.263158          19
5  2026-06-04      -0.095238          21
6  2026-06-05      -0.142857          21
7  2026-06-07       0.062500          16
8  2026-06-08       0.000000          20
9  2026-06-09       0.052632          19
10 2026-06-10       0.142857          21
11 2026-06-11      -0.153846          13
12 2026-06-12       0.055556          18
13 2026-06-14       0.181818          11
14 2026-06-15       0.136364          22
15 2026-06-16       0.000000          23
16 2026-06-17      -0.043478          23
17 2026-06-18       0.062500          16
18 2026-06-19       0.136364          22
19 2026-06-21       0.666667          12
20 2026-06-22       0.000000          17
21 2026-06-23       0.062500          16
22 2026-06-24       0.000000          22
23 2026-06-25   